# FL/OD Analysis
Fluorescence normalized by fl, grouped by light condition and intensity.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ── Load data ──────────────────────────────────────────────────
od_df = pd.read_csv('od.csv')
fl_df = pd.read_csv('fl.csv')

# ── Time → hours ───────────────────────────────────────────────
def to_hours(t):
    h, m, s = t.split(':')
    return int(h) + int(m)/60 + int(s)/3600

time_h = fl_df['Time'].apply(to_hours).values

# ── PID well groups (matches controller's pid1..pid10) ─────────
conditions = {
    'Red':       ['A3','A4','A6'],
    'Green':     ['B3','B4','B6'],
    'PID1_SP1':  ['C1','D1','E1'],
    'PID2_SP1':  ['F1','G1','H1'],
    'PID3_SP1':  ['C3','C4','C6'],
    'PID4_SP1':  ['D3','D4','D6'],
    'PID1_SP2':  ['E3','E4','E6'],
    'PID2_SP2':  ['F3','F4','F6'],
    'PID3_SP2':  ['G3','G4','G6'],
    'PID4_SP2':  ['H3','H4','H6'],
}

NEG_WELLS = ['A1','B1']
OD_BLANK = 0.1

# ── Subtract blanks ────────────────────────────────────────────
all_wells = NEG_WELLS + [w for ws in conditions.values() for w in ws]
od    = od_df[all_wells].astype(float)
fl    = fl_df[all_wells].astype(float)
fl_od = fl / od
neg_fl_od = fl_od[NEG_WELLS].mean(axis=1).values

# ── Shared colour map ────────────────────────────────────────
# Red/Green light conditions get their literal colours.
# Each PID gain group (1-4) gets ONE fixed colour shared by both
# its setpoints (SP1 and SP2) — e.g. PID1_SP1 and PID1_SP2 are the
# same colour since they use the same gain values, just different
# setpoints. Setpoint is instead distinguished by linestyle
# (solid = SP1, dashed = SP2).
pid_shades = {
    'Red':   'red',
    'Green': 'green',
    'PID1_SP1': 'tab:blue',    'PID1_SP2': 'tab:blue',
    'PID2_SP1': 'tab:gray',    'PID2_SP2': 'tab:gray',
    'PID3_SP1': 'tab:pink',    'PID3_SP2': 'tab:pink',
    'PID4_SP1': 'tab:brown',   'PID4_SP2': 'tab:brown',
}

# Linestyle map: solid for SP1 groups / Red / Green, dashed for SP2 groups.
pid_linestyles = {
    cond: ('--' if cond.endswith('SP2') else '-')
    for cond in conditions
}

# Per-well style lookup (color, group label, linestyle), used by the
# column-grid plots in Cells 5-7. This used to only exist as a
# commented-out block inside Cell 5, which meant 'well_style' was
# undefined and Cells 5-7 raised NameError when run.
well_style = {}
for cond, wells in conditions.items():
    color = pid_shades[cond]
    ls = pid_linestyles[cond]
    for w in wells:
        well_style[w] = (color, cond, ls)

print('Setup complete.')

In [ ]:
# ── Cell 2: FL/OD plot, all PID groups ──────────────────────────
results = {}
for cond, wells in conditions.items():
    vals = fl_od[wells].values - neg_fl_od[:, np.newaxis]
    mean = vals.mean(axis=1)
    se   = vals.std(axis=1, ddof=1) / np.sqrt(vals.shape[1]) if vals.shape[1] > 1 else np.zeros(len(mean))
    results[cond] = (mean, se)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
fig.suptitle('FL/OD by PID Group', fontsize=13, fontweight='bold')

for cond, color in pid_shades.items():
    mean, se = results[cond]
    ax.plot(time_h, mean, color=color, label=f'{cond} (n={len(conditions[cond])})', linewidth=1.5)
    ax.fill_between(time_h, mean - se, mean + se, color=color, alpha=0.2)

ax.axhline(y=11500, color='purple', linestyle='-', linewidth=1.2)
ax.axhline(y=18500, color='orange', linestyle='-', linewidth=1.2)

ax.set_xlim(0, 16)
ax.set_ylim(0,30000)
ax.set_xlabel('Time (hours)', fontsize=13)
ax.set_ylabel('FL / OD (a.u.)', fontsize=13)
handles, labels = ax.get_legend_handles_labels()
handles, labels = zip(*sorted(zip(handles, labels), key=lambda t: t[1]))
ax.legend(handles, labels, fontsize=10, loc='upper left', frameon=False)
ax.grid(True, alpha=0.3)
ax.tick_params(labelsize=13)

plt.tight_layout()
plt.savefig('fl_od_figure.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 3: OD plot (individual wells) ────────────────────────
fig, ax = plt.subplots(figsize=(7.5, 4.5))
fig.suptitle(r'$OD_{600}$', fontsize=16, fontweight='bold')

seen = set()

for cond, color in pid_shades.items():
    for well in conditions[cond]:
        ax.plot(time_h, od[well].values-OD_BLANK, color=color, linewidth=1,
                label=cond if cond not in seen else '_')
        seen.add(cond)

ax.plot(time_h, od[NEG_WELLS].values-OD_BLANK, color='black', linewidth=1, label='Neg Cont.')

ax.set_xlim(0, 16)
ax.set_ylim(0,1.1)
ax.set_xlabel('Time (hours)', fontsize=15)
ax.set_ylabel(r'$OD_{600}$', fontsize=15)
handles, labels = ax.get_legend_handles_labels()
handles, labels = zip(*sorted(zip(handles, labels), key=lambda t: t[1]))
ax.legend(handles, labels, fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3)
ax.tick_params(labelsize=12)

plt.tight_layout()
plt.savefig('od_figure.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 4: Raw FL plot (individual wells) ─────────────────────
fig, ax = plt.subplots(figsize=(7.5, 4.5))
fig.suptitle('Raw Fluorescence (mCherry)', fontsize=16, fontweight='bold')

seen = set()

for cond, color in pid_shades.items():
    for well in conditions[cond]:
        ax.plot(time_h, fl[well].values, color=color, linewidth=1,
                label=cond if cond not in seen else '_')
        seen.add(cond)

ax.plot(time_h, fl[NEG_WELLS].values, color='black', linewidth=1, label='NC')

ax.set_xlim(0, 16)
ax.set_ylim(0,35000)
ax.set_xlabel('Time (hours)', fontsize=15)
ax.set_ylabel('FL (a.u.)', fontsize=15)
handles, labels = ax.get_legend_handles_labels()
handles, labels = zip(*sorted(zip(handles, labels), key=lambda t: t[1]))
ax.legend(handles, labels, fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3)
ax.tick_params(labelsize=12)

plt.tight_layout()
plt.savefig('fl_figure.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 5: OD by column (2x2 grid) ───────────────────────────
col_wells = {
    'Column 1': ['A1','B1','C1','D1','E1','F1','G1','H1'],
    'Column 3': ['A3','B3','C3','D3','E3','F3','G3','H3'],
    'Column 4': ['A4','B4','C4','D4','E4','F4','G4','H4'],
    'Column 6': ['A6','B6','C6','D6','E6','F6','G6','H6'],
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)
fig.suptitle(r'$OD_{600}$ by Column', fontsize=14, fontweight='bold')

for ax, (col_name, wells) in zip(axes.flatten(), col_wells.items()):
    seen = set()
    for well in wells:
        if well not in well_style:
            continue
        color, label, ls = well_style[well]
        ax.plot(time_h, od[well].values-OD_BLANK, color=color, linestyle=ls, linewidth=1.5,
                label=label if label not in seen else '_')
        seen.add(label)
    ax.set_xlim(0, 16); ax.set_ylim(0,1.1)
    ax.set_title(col_name, fontsize=12)
    ax.set_xlabel('Time (hours)', fontsize=11)
    handles, labels = ax.get_legend_handles_labels()
    handles, labels = zip(*sorted(zip(handles, labels), key=lambda t: t[1]))
    ax.legend(handles, labels, fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3)

axes[0,0].set_ylabel(r'$OD_{600}$', fontsize=12)
axes[1,0].set_ylabel(r'$OD_{600}$', fontsize=12)
plt.tight_layout()
plt.subplots_adjust(hspace=0.4)
# plt.savefig('od_by_column.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 6: FL by column (2x2 grid) ───────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)
fig.suptitle('FL by Column', fontsize=14, fontweight='bold')

for ax, (col_name, wells) in zip(axes.flatten(), col_wells.items()):
    seen = set()
    for well in wells:
        if well not in well_style:
            continue
        color, label, ls = well_style[well]
        ax.plot(time_h, fl[well].values, color=color, linestyle=ls, linewidth=1.5,
                label=label if label not in seen else '_')
        seen.add(label)
    ax.set_xlim(0, 16); ax.set_ylim(0,40000)
    ax.set_title(col_name, fontsize=12)
    ax.set_xlabel('Time (hours)', fontsize=11)
    handles, labels = ax.get_legend_handles_labels()
    handles, labels = zip(*sorted(zip(handles, labels), key=lambda t: t[1]))
    ax.legend(handles, labels, fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3)

axes[0,0].set_ylabel('FL (a.u.)', fontsize=12)
axes[1,0].set_ylabel('FL (a.u.)', fontsize=12)
plt.tight_layout()
plt.subplots_adjust(hspace=0.4)
# plt.savefig('fl_by_column.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 7: FL_OD by column (2x2 grid) ───────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)
fig.suptitle('FL/OD by Column', fontsize=14, fontweight='bold')

for ax, (col_name, wells) in zip(axes.flatten(), col_wells.items()):
    seen = set()
    for well in wells:
        if well not in well_style:
            continue
        color, label, ls = well_style[well]
        ax.plot(time_h, fl_od[well].values, color=color, linestyle=ls, linewidth=1.5,
                label=label if label not in seen else '_')
        seen.add(label)
    ax.set_xlim(0, 16); ax.set_ylim(0,40000)
    ax.set_title(col_name, fontsize=12)
    ax.set_xlabel('Time (hours)', fontsize=11)
    handles, labels = ax.get_legend_handles_labels()
    handles, labels = zip(*sorted(zip(handles, labels), key=lambda t: t[1]))
    ax.legend(handles, labels, fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3)

axes[0,0].set_ylabel('FL/OD (a.u.)', fontsize=12)
axes[1,0].set_ylabel('FL/OD (a.u.)', fontsize=12)
plt.tight_layout()
plt.subplots_adjust(hspace=0.4)
plt.savefig('fl_od_by_column.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 8: Export FL/OD for all wells to CSV ────────────
export_wells = [w for ws in conditions.values() for w in ws]  # all 30 PID wells, pid1..pid10

export_df = pd.DataFrame({'Time_min': time_h*60})
for well in export_wells:
    export_df[well] = fl_od[well].values

output_path = 'fl_od_all_wells.csv'
export_df.to_csv(output_path, index=False)
print(f'Exported FL/OD for {len(export_wells)} wells to "{output_path}"')
export_df.head()